# Supermarket Sales Analysis
## Data Analytics Project

**Objective:** Analyze supermarket sales data to discover useful information about products, branches, categories, customers, payment methods, and customer ratings.

### Problem Statement
The aim of this project is to analyze supermarket sales data and find useful information about products, branches, categories, customers, payments, and ratings.

### Dataset
The dataset contains **500 sales transactions** with information such as invoice ID, date, branch, city, customer type, gender, product, category, quantity, unit price, payment method, rating, and sales.

### Analysis Workflow
1. Collect and load the CSV dataset.
2. Check the data for missing or incorrect values.
3. Calculate **Sales = Quantity × Unit Price**.
4. Group and summarize the data using totals, counts, and averages.
5. Create charts to compare the results.
6. Use the results to make business decisions.


In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Display plots inside the notebook
%matplotlib inline

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


In [ ]:
# Load the supermarket sales CSV dataset
file_path = "SUPER MARKET DATA - supermarket_sales_500_rows.csv"

df = pd.read_csv(file_path)

print(f"Dataset shape: {df.shape}")
display(df.head())


## 1. Dataset Overview

In [ ]:
# Basic information about the dataset
print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("Data Type"))

print("\nDescriptive statistics:")
display(df.describe(include="all").T)


## 2. Data Quality Check
Check for missing values, duplicate rows, invalid quantities/prices/ratings, and verify the sales calculation.

In [ ]:
# Missing values
missing_values = df.isna().sum().sort_values(ascending=False)
display(missing_values.to_frame("Missing Values"))

# Duplicate rows
print("Duplicate rows:", df.duplicated().sum())

# Basic validity checks
print("Rows with quantity <= 0:", (df["Quantity"] <= 0).sum())
print("Rows with unit price <= 0:", (df["Unit Price"] <= 0).sum())
print("Ratings outside 1-5:", ((df["Rating"] < 1) | (df["Rating"] > 5)).sum())

# Convert Date to datetime
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
print("Invalid dates after conversion:", df["Date"].isna().sum())


## 3. Calculate and Verify Sales
The project specification defines **Sales = Quantity × Unit Price**.

In [ ]:
# Calculate sales independently from quantity and unit price
df["Calculated Sales"] = df["Quantity"] * df["Unit Price"]

# Compare calculated sales with the Sales column
df["Sales Difference"] = (df["Sales"] - df["Calculated Sales"]).abs()

print("Maximum sales calculation difference:",
      df["Sales Difference"].max())

print("Rows where sales does not match Quantity × Unit Price:",
      (df["Sales Difference"] > 0.01).sum())

# Remove helper columns after verification
df.drop(columns=["Calculated Sales", "Sales Difference"], inplace=True)

display(df.head())


## 4. Summary Statistics

In [ ]:
summary = pd.DataFrame({
    "Metric": [
        "Total Transactions",
        "Total Sales",
        "Average Transaction Sales",
        "Total Quantity Sold",
        "Average Customer Rating"
    ],
    "Value": [
        len(df),
        df["Sales"].sum(),
        df["Sales"].mean(),
        df["Quantity"].sum(),
        df["Rating"].mean()
    ]
})

display(summary)


## 5. Product-wise Sales Analysis

In [ ]:
product_sales = (
    df.groupby("Product", as_index=False)["Sales"]
      .sum()
      .sort_values("Sales", ascending=False)
)

display(product_sales)

top_product = product_sales.iloc[0]
print(f"Highest-selling product: {top_product['Product']}")
print(f"Sales: ₹{top_product['Sales']:,.2f}")


In [ ]:
# Product sales chart
plt.figure(figsize=(10, 5))
plt.bar(product_sales["Product"], product_sales["Sales"])
plt.title("Sales by Product")
plt.xlabel("Product")
plt.ylabel("Sales (₹)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 6. Branch-wise Sales Analysis

In [ ]:
branch_sales = (
    df.groupby(["Branch", "City"], as_index=False)["Sales"]
      .sum()
      .sort_values("Sales", ascending=False)
)

display(branch_sales)

top_branch = branch_sales.iloc[0]
print(f"Best-performing branch: Branch {top_branch['Branch']} ({top_branch['City']})")
print(f"Sales: ₹{top_branch['Sales']:,.2f}")


In [ ]:
# Branch sales chart
labels = branch_sales["Branch"] + " (" + branch_sales["City"] + ")"

plt.figure(figsize=(9, 5))
plt.bar(labels, branch_sales["Sales"])
plt.title("Sales by Branch")
plt.xlabel("Branch")
plt.ylabel("Sales (₹)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 7. Category-wise Sales Analysis

In [ ]:
category_sales = (
    df.groupby("Category", as_index=False)["Sales"]
      .sum()
      .sort_values("Sales", ascending=False)
)

display(category_sales)

top_category = category_sales.iloc[0]
print(f"Highest-selling category: {top_category['Category']}")
print(f"Sales: ₹{top_category['Sales']:,.2f}")


In [ ]:
# Category sales chart
plt.figure(figsize=(10, 5))
plt.bar(category_sales["Category"], category_sales["Sales"])
plt.title("Sales by Category")
plt.xlabel("Category")
plt.ylabel("Sales (₹)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 8. Payment Method Analysis

In [ ]:
payment_counts = (
    df["Payment"]
      .value_counts()
      .rename_axis("Payment Method")
      .reset_index(name="Transactions")
)

display(payment_counts)

top_payment = payment_counts.iloc[0]
print(f"Most popular payment method: {top_payment['Payment Method']}")
print(f"Transactions: {top_payment['Transactions']}")


In [ ]:
# Payment method chart
plt.figure(figsize=(8, 5))
plt.bar(payment_counts["Payment Method"], payment_counts["Transactions"])
plt.title("Transactions by Payment Method")
plt.xlabel("Payment Method")
plt.ylabel("Number of Transactions")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 9. Member vs Normal Customer Spending

In [ ]:
customer_spending = (
    df.groupby("Customer Type")["Sales"]
      .mean()
      .sort_values(ascending=False)
)

display(customer_spending.to_frame("Average Transaction Sales"))

member_avg = customer_spending.get("Member")
normal_avg = customer_spending.get("Normal")

print(f"Average Member transaction: ₹{member_avg:,.2f}")
print(f"Average Normal transaction: ₹{normal_avg:,.2f}")

if member_avg > normal_avg:
    print("Members have a higher average transaction value.")
elif member_avg < normal_avg:
    print("Normal customers have a higher average transaction value.")
else:
    print("Both customer types have the same average transaction value.")


In [ ]:
# Average spending by customer type
plt.figure(figsize=(7, 5))
plt.bar(customer_spending.index, customer_spending.values)
plt.title("Average Transaction Sales by Customer Type")
plt.xlabel("Customer Type")
plt.ylabel("Average Sales (₹)")
plt.tight_layout()
plt.show()


## 10. Customer Rating Analysis

In [ ]:
average_rating = df["Rating"].mean()

print(f"Average customer rating: {average_rating:.2f} out of 5")

rating_summary = (
    df["Rating"]
      .round(1)
      .value_counts()
      .sort_index()
      .rename_axis("Rating")
      .reset_index(name="Number of Customers")
)

display(rating_summary)


In [ ]:
# Customer rating distribution
plt.figure(figsize=(9, 5))
plt.bar(rating_summary["Rating"].astype(str), rating_summary["Number of Customers"])
plt.title("Customer Rating Distribution")
plt.xlabel("Rating")
plt.ylabel("Number of Transactions")
plt.tight_layout()
plt.show()


## 11. Key Business Questions — Live Analysis

In [ ]:
# Answer the required business questions directly from the dataset

# 1. Which product generates the highest sales?
highest_product = df.groupby("Product")["Sales"].sum().idxmax()
highest_product_sales = df.groupby("Product")["Sales"].sum().max()

# 2. Which branch performs best?
branch_totals = df.groupby(["Branch", "City"])["Sales"].sum()
best_branch_city = branch_totals.idxmax()
best_branch_sales = branch_totals.max()

# 3. Which category sells the most?
highest_category = df.groupby("Category")["Sales"].sum().idxmax()
highest_category_sales = df.groupby("Category")["Sales"].sum().max()

# 4. What is the most popular payment method?
most_popular_payment = df["Payment"].value_counts().idxmax()
most_popular_payment_count = df["Payment"].value_counts().max()

# 5. Do Members spend more than Normal customers?
member_average = df.loc[df["Customer Type"] == "Member", "Sales"].mean()
normal_average = df.loc[df["Customer Type"] == "Normal", "Sales"].mean()

# 6. What is the average customer rating?
average_rating = df["Rating"].mean()

print(f"1. Highest-sales product: {highest_product} — ₹{highest_product_sales:,.2f}")
print(f"2. Best branch: Branch {best_branch_city[0]} ({best_branch_city[1]}) — ₹{best_branch_sales:,.2f}")
print(f"3. Highest-sales category: {highest_category} — ₹{highest_category_sales:,.2f}")
print(f"4. Most popular payment method: {most_popular_payment} — {most_popular_payment_count} transactions")
print(f"5. Average Member transaction: ₹{member_average:,.2f}")
print(f"   Average Normal transaction: ₹{normal_average:,.2f}")
print(f"6. Average customer rating: {average_rating:.2f} / 5")


## 12. Business Decisions

Based on the analysis:

- Keep sufficient stock of high-selling products and categories, especially the strongest-selling items.
- Study the factors contributing to **Branch C (Mumbai)** having the highest sales.
- Continue supporting **UPI**, which is the most-used payment method in this dataset.
- Customer ratings can be monitored to identify opportunities for improving customer service.
- The difference between Member and Normal average transaction values can be considered when planning membership offers.


## 13. Conclusion

This analysis shows how supermarket sales data can be converted into practical business insights. By grouping sales by product, branch, category, customer type, and payment method, and by analyzing customer ratings, the supermarket can better understand its performance and identify areas for operational improvement.

### Final Results
- **Highest product sales:** Cheese — **₹27,906.30**
- **Best-performing branch:** Branch C (Mumbai) — **₹72,469.45**
- **Highest category sales:** Beverages — **₹56,108.24**
- **Most-used payment method:** UPI — **127 transactions**
- **Average Member transaction:** **₹483.14**
- **Average Normal transaction:** **₹497.07**
- **Average customer rating:** **3.99 / 5**
